# Phase 1 - Environment Smoke Test

This notebook verifies the services used by the original-data ingestion demo: MinIO object storage and Kafka streaming. Credentials come from `.env`; no credentials are embedded in the notebook.

In [1]:
import os
import sys
import boto3
import pandas as pd
from kafka import KafkaAdminClient

required_env = [
    "MINIO_ENDPOINT_INTERNAL", "MINIO_ROOT_USER", "MINIO_ROOT_PASSWORD",
    "RAW_BUCKET", "LAKEHOUSE_BUCKET", "KAFKA_BOOTSTRAP_SERVERS",
]
missing = [name for name in required_env if not os.getenv(name)]
assert not missing, f"Missing environment variables: {missing}"
print("Python:", sys.version.split()[0])
print("Required environment variables are available.")

Python: 3.13.13
Required environment variables are available.


## MinIO Buckets

The raw bucket stores immutable source files. The lakehouse bucket stores processed Bronze, Silver, and Gold tables.

In [2]:
s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["MINIO_ENDPOINT_INTERNAL"],
    aws_access_key_id=os.environ["MINIO_ROOT_USER"],
    aws_secret_access_key=os.environ["MINIO_ROOT_PASSWORD"],
    region_name=os.getenv("AWS_REGION", "us-east-1"),
)
buckets = [entry["Name"] for entry in s3.list_buckets()["Buckets"]]
print("MinIO buckets:", buckets)
assert os.environ["RAW_BUCKET"] in buckets
assert os.environ["LAKEHOUSE_BUCKET"] in buckets

MinIO buckets: ['lakehouse', 'mlflow', 'raw', 'warehouse']


## Kafka Connectivity

Kafka is the streaming-ingestion layer. The replay notebook sends original ARCO-ERA5 records to `weather.raw`.

In [3]:
admin = KafkaAdminClient(
    bootstrap_servers=os.environ["KAFKA_BOOTSTRAP_SERVERS"],
    client_id="aviation-phase1-smoke-test",
)
print("Kafka topics:", sorted(admin.list_topics()))
admin.close()
print("Phase 1 service smoke test complete.")

Kafka topics: ['__consumer_offsets', 'weather.raw']
Phase 1 service smoke test complete.
